# VRC-3PO — IEEE Access revision (Access-2026-38295), round 1 experiments

Use a GPU runtime: **Runtime → Change runtime type → T4 GPU**. Run the cells in order. Every training step is **resumable**: after a disconnect, rerun the setup cells (1–3) and then the step you were on; finished folds print `[resume]` and are skipped.

| Step | Reviewer comment | What runs | Approx. T4 time |
|---|---|---|---|
| 1 | R2-7 preprocessing | Build and verify **benchmark v1.1** from the raw recordings | 20–40 min (Maze) |
| 2 | — | Smoke test | 3 min |
| 3 | R2-1 headline | 5-CNN ensemble, **5×5 repeated participant CV** + reference split | 1–1.5 h |
| 4 | R2-2, R1-3 | Environment-only / source-only baselines + feature-based baselines | 5 min (CPU) |
| 5 | R2-3 threshold | Ensemble under 3 source-aware label definitions (3×5 CV) | 2–3 h |
| 6 | R2-6 arch × composition | CNN, CNN+TCN, VRC-3PO, MLP on the **same** 20 CV compositions | 1.5–2 h |
| 7 | R2-5 tuning | Equal-budget (8-trial) validation-only tuning, 4 architectures | 2–3 h |
| 8 | R2-7 rerun | Submitted analyses (42-fit architecture table, 20 compositions) on v1.1 | 3–4 h |
| 9 | all | Summaries → `results_revision.zip` | 5–10 min |

Steps 3–8 are independent after Step 1, so they can run in separate Colab sessions.

In [ ]:
# ═══ 1 · Runtime ═══
!pip install -q keras-tcn
import tensorflow as tf, pandas as pd, numpy as np
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ═══ 2 · Drive + code bundle ═══
from google.colab import drive, files
from pathlib import Path
import zipfile, shutil, os, subprocess
drive.mount('/content/drive')

BUNDLE = Path('/content/drive/MyDrive/vrc3po_revision/vrc3po_revision_bundle.zip')
if not BUNDLE.exists():
    print('Upload vrc3po_revision_bundle.zip (it is also kept on Drive for later sessions).')
    up = files.upload()
    name = [n for n in up if n.endswith('.zip')][0]
    BUNDLE.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(name, BUNDLE)
root = Path('/content/vrc3po_code')
shutil.rmtree(root, ignore_errors=True); root.mkdir()
zipfile.ZipFile(BUNDLE).extractall(root)
os.chdir(root)
!pip install -q pytest
!python -m pytest tests -q

In [ ]:
# ═══ 3 · Paths ═══
from pathlib import Path
RAW   = '/content/drive/MyDrive/utsa_eye_head_cybersickness_datasets'
REV   = Path('/content/drive/MyDrive/vrc3po_revision')
STAGE = REV / 'v11_stage'
V11   = REV / 'vrc3po_benchmark_v1_1.csv'
V10   = '/content/drive/MyDrive/vrc3po_master_dataset_fixed.csv'   # submitted table, for comparison
OUT   = REV / 'cv'                                                   # all repeated-CV runs
for p in (REV, STAGE, OUT): p.mkdir(parents=True, exist_ok=True)
print('raw data found:', Path(RAW).exists(), '| v1.0 found:', Path(V10).exists(), '| v1.1 built:', V11.exists())

## Step 1 · Benchmark v1.1 (R2-7)
Replaces the earlier stand-alone M0 notebook: same corrections, now as `analysis/benchmark_v11.py` inside the release archive, and with the published participant-ID format (`maze_1.0` …) so the frozen seed-42 split can be re-derived. Maze is staged per participant and resumes if interrupted; rerun the cell until it finishes.

In [ ]:
# ═══ Step 1 · build, verify, compare ═══
if not V11.exists():
    !python -m analysis.benchmark_v11 stage --raw-dir "$RAW" --stage-dir "$STAGE" --source terrain
    !python -m analysis.benchmark_v11 stage --raw-dir "$RAW" --stage-dir "$STAGE" --source simulations
    !python -m analysis.benchmark_v11 stage --raw-dir "$RAW" --stage-dir "$STAGE" --source maze
    !python -m analysis.benchmark_v11 finalize --stage-dir "$STAGE" --output "$V11"
!python -m analysis.benchmark_v11 verify "$V11"

import json
from analysis.revision_common import build_windows
man = json.loads(V11.with_suffix('.manifest.json').read_text())
print(json.dumps({k: man[k] for k in ('rows','participants','recordings','rows_by_source','simulations_native_rate_masking')}, indent=2))
print('Removed recordings:'); display(pd.DataFrame(man['recordings_removed']))
w = {}
for tag, path in (('v1.0', V10), ('v1.1', V11)):
    if Path(path).exists():
        _, fms, meta = build_windows(pd.read_csv(path))
        meta['elevated'] = fms > 2
        w[tag] = meta.groupby(['source_dataset','condition']).agg(windows=('elevated','size'), elevated=('elevated','sum'))
display(pd.concat(w, axis=1))

## Step 2 · Smoke test (3 folds, 1 epoch)

In [ ]:
!python -m analysis.revision_cv run --dataset "$V11" --output-dir "$REV/smoke" --models cnn_ensemble feat:logreg nuisance:condition --repeats 1 --folds 3 --epochs 1
!python -m analysis.revision_cv summarize --dataset "$V11" --output-dir "$REV/smoke" --bootstrap 100
print('Smoke test passed.')

## Step 3 · Primary estimate (R2-1)
Stratified participant-level 5-fold CV repeated 5 times over the 47 seated participants: every participant is tested exactly once per repeat, with the frozen 5-CNN ensemble recipe. The mean over repeats with a participant-cluster bootstrap CI becomes the headline. The frozen seed-42 split is rerun on v1.1 as a single benchmark realization.

In [ ]:
assert tf.config.list_physical_devices('GPU'), 'Switch to a GPU runtime.'
!python -m analysis.revision_cv run --dataset "$V11" --output-dir "$OUT" --models cnn_ensemble --labels fms_gt2 --repeats 5 --folds 5
!python -m analysis.revision_cv run --dataset "$V11" --output-dir "$OUT" --models cnn_ensemble --labels fms_gt2 --protocol reference

## Step 4 · Environment control and baselines (R2-2, R1-3) — CPU, minutes
* `nuisance:condition` / `nuisance:source`: logistic regression on environment or source identity only, with no physiology. This is the environment-only baseline the reviewer asked for.
* `feat:logreg | rf | hgb`: conventional feature-based detectors (84 per-window statistics: mean, SD, min, max, slope and change of each channel).
The summary adds the **within-condition AUC** (pairs from the same environment and fold only, so it is exactly 0.5 for any environment-only score) and the **incremental AUC** of each model over environment identity.

In [ ]:
!python -m analysis.revision_cv run --dataset "$V11" --output-dir "$OUT" --models nuisance:condition nuisance:source feat:logreg feat:rf feat:hgb --labels fms_gt2 --repeats 5 --folds 5

## Step 5 · Threshold sensitivity (R2-3)
| label | rule | Simulations | Terrain |
|---|---|---|---|
| `fms_gt2` | published | > 2 | > 2 |
| `floor_plus2` | > 2 above scale floor | > 2 | > 3 |
| `floor_plus1` | > 1 above scale floor | > 1 | > 2 |
| `baseline_plus1` | > 1 above the recording's first rating | individual | individual |

In [ ]:
!python -m analysis.revision_cv run --dataset "$V11" --output-dir "$OUT" --models cnn_ensemble nuisance:condition --labels floor_plus2 floor_plus1 baseline_plus1 --repeats 3 --folds 5

## Step 6 · Architectures on the same compositions (R2-6)
Four representative bodies from the architecture table, each with a sigmoid output and the ensemble's training recipe (BCE, balanced class weights, early stopping on validation AUC), trained on the **same 20 folds** (4 repeats × 5) as Step 3. The summary decomposes fold-level AUC variance into composition vs architecture.

In [ ]:
!python -m analysis.revision_cv run --dataset "$V11" --output-dir "$OUT" --models clf:MLP clf:CNN "clf:CNN+TCN" clf:VRC-3PO --labels fms_gt2 --repeats 4 --folds 5

## Step 7 · Equal-budget validation-only tuning (R2-5)
Every architecture gets the same 8 random trials over learning rate × batch size × weight decay per fold, ranked by **validation-participant** AUC. Only the selected trial is scored on the test participants. Layer stacks are unchanged, so the architecture table stays valid. Can be skipped if we choose to qualify the claim instead.

In [ ]:
!python -m analysis.revision_cv run --dataset "$V11" --output-dir "$OUT" --models tuned:MLP tuned:CNN "tuned:CNN+TCN" tuned:VRC-3PO --labels fms_gt2 --repeats 1 --folds 5 --trials 8

## Step 8 · Submitted analyses rerun on v1.1 (R2-7)
The 42-fit architecture / protocol table and the 20-composition study, unchanged except for the cleaned input.

In [ ]:
!python -m analysis.retrain_camera_ready --dataset "$V11" --output-dir "$REV/camera_ready_v11"
!python -m analysis.composition_robustness --dataset "$V11" --output-dir "$REV/composition_v11" --compositions 20

## Step 9 · Summaries → `results_revision.zip`

In [ ]:
!python -m analysis.revision_cv summarize --dataset "$V11" --output-dir "$OUT" --bootstrap 2000
S = OUT / 'summary'
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
p = pd.read_csv(S / 'primary_estimates.csv')
print('PRIMARY (fms_gt2):'); display(p[p.label == 'fms_gt2'].round(3))
print('THRESHOLD SENSITIVITY:'); display(p[p.model == 'cnn_ensemble'].round(3))
for f in ('environment_control', 'paired_comparisons', 'operating_points_validation_selected', 'per_condition_auc'):
    print(f.upper()); display(pd.read_csv(S / f'{f}.csv').round(3))
print((S / 'variance_components.json').read_text())

import shutil
bundle = Path('/content/results_revision')
shutil.rmtree(bundle, ignore_errors=True); bundle.mkdir()
shutil.copytree(S, bundle / 'cv_summary')
for sub in ('camera_ready_v11', 'composition_v11'):
    src = REV / sub
    if src.exists():
        shutil.copytree(src, bundle / sub, ignore=shutil.ignore_patterns('*.npz', 'completed_runs', '*predictions*'))
shutil.copy(V11.with_suffix('.manifest.json'), bundle)
shutil.make_archive('/content/results_revision', 'zip', bundle)
shutil.copy('/content/results_revision.zip', REV / 'results_revision.zip')
print('Saved to Drive:', REV / 'results_revision.zip')
print('Copy it into the Sickness-Is-In-The-Eye-of-the-Beholder/results/ folder on your Mac (no browser download needed).')